In [1]:
import os
import pandas as pd
import json
import kagglehub

# 1. Download dataset
print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("himanshunayal/intent-recognition-dataset")
print(f"Path to dataset files: {path}")

# 2. Find the CSV file inside the downloaded folder
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
if not csv_files:
    print("Error: No CSV file found in the downloaded dataset.")
    exit()

csv_path = os.path.join(path, csv_files[0])
print(f"Reading data from: {csv_files[0]}")

# 3. Load the data using Pandas
df = pd.read_csv(csv_path)

text_col = df.columns[0]
intent_col = df.columns[1]

# 4. Group all user phrases by their specific intent
grouped_data = df.groupby(intent_col)[text_col].apply(list).to_dict()

# 5. Build the exact JSON structure
new_dataset = {"intents": []}

for intent_name, patterns in grouped_data.items():
    clean_patterns = [str(p) for p in patterns] 
    
    intent_data = {
        "tag": str(intent_name),
        "patterns": clean_patterns[:20],
        "responses": [
            f"[System]: Intent recognized as '{intent_name}'. I need a developer to code my actual response for this!", 
            f"[System]: You triggered the '{intent_name}' category."
        ]
    }
    new_dataset["intents"].append(intent_data)

# ---> FIX 3: INJECT CUSTOM GREETING INTENT <---
# ---> REPLACE THE OLD GREETING INJECTION WITH THIS WHOLE BLOCK <---

# 1. Greetings
new_dataset["intents"].append({
    "tag": "Greeting",
    "patterns": [
        "hello", "hi", "hey", "how are you", "good morning", "what's up", "hiya", "hey there",
        "hello chatbot", "hi bot", "yo", "wassup", "sup", "hello there", "good afternoon"
    ],
    "responses": ["Hello! How can I assist you today?", "Hi there!", "Greetings!"]
})

# 2. Goodbyes
new_dataset["intents"].append({
    "tag": "Goodbye",
    "patterns": [
        "bye", "goodbye", "see you later", "i am leaving", "talk to you later", 
        "catch you later", "farewell", "bye bye", "exit", "quit"
    ],
    "responses": ["Goodbye!", "Have a great day!", "See you later!"]
})

# 3. Thanks
new_dataset["intents"].append({
    "tag": "Thanks",
    "patterns": [
        "thanks", "thank you", "that helps", "awesome thanks", "thank you so much",
        "perfect thank you", "much appreciated", "thx"
    ],
    "responses": ["You're very welcome!", "Anytime!", "Glad I could help!"]
})

# 4. Bot Identity / Wellbeing
new_dataset["intents"].append({
    "tag": "Bot_Status",
    "patterns": [
        "how are you", "how are you doing", "how's it going", "are you good",
        "who are you", "what is your name", "what are you"
    ],
    "responses": ["I'm doing great, thank you for asking!", "I'm your AI assistant, ready to help!"]
})
# 6. Save the new intents.json file
with open('intents.json', 'w', encoding='utf-8') as f:
    json.dump(new_dataset, f, indent=4)

print(f"\n✅ Success! Converted {len(grouped_data) + 1} different intents into a new 'intents.json' file.")
print("Run your chatbot script now—it will automatically load all this new data!")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /root/.cache/kagglehub/datasets/himanshunayal/intent-recognition-dataset/versions/1
Reading data from: test.csv

✅ Success! Converted 8 different intents into a new 'intents.json' file.
Run your chatbot script now—it will automatically load all this new data!


In [ ]:
import json
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from googletrans import Translator
from sklearn.feature_extraction.text import TfidfVectorizer
# 1. Load the data
with open('intents.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Prepare the training data
patterns = []
tags = []
for intent in data['intents']:
    for pattern in intent['patterns']:
        patterns.append(pattern)
        tags.append(intent['tag'])

# 3. Train the model
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(patterns)
model = MultinomialNB()
model.fit(X, tags)

translator = Translator()

print("Chatbot initialized and training complete!")

# 4. Chat Loop
while True:
    user_input = input("\nYou: ")
    if user_input.lower() == 'bye': 
        print("Bot: Goodbye!")
        break

    # Translate to English for processing
    processed_input = (await translator.translate(user_input, dest='en')).text

    # ---> FIX 2: PREDICT INTENT WITH CONFIDENCE THRESHOLD <---
    input_vector = vectorizer.transform([processed_input])
    probabilities = model.predict_proba(input_vector)[0]
    
    # Get the highest probability and its corresponding tag
    max_prob = max(probabilities)
    prediction = model.classes_[probabilities.argmax()]

    # If the bot is less than 30% sure, fallback to a default message
    if max_prob < 0.10:
        reply = "I'm sorry, I don't understand that command."
    else:
        # Find response normally
        reply = "I'm not sure about that."
        for intent in data['intents']:
            if intent['tag'] == prediction:
                reply = random.choice(intent['responses'])
                break

    # Translate reply back
    final_reply = (await translator.translate(reply, dest='en')).text 
    print(f"Bot: {final_reply}")

Chatbot initialized and training complete!



You:  hello


Bot: Hi there!



You:  how are you


Bot: I'm your AI assistant, ready to help!
